# Group name: Janita&Angelina
#  

In [1]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from scipy import interpolate 
import os, re
from scipy.signal.windows import triang
from scipy.signal import find_peaks, butter, filtfilt, sosfiltfilt, medfilt

# --- 1. Load ECG Data with HR Labels ---

In [2]:
file_path = os.path.join("data", "ecg_data_with_hr_labels.pkl")  

# Load ECG data
with open(file_path, "rb") as f:
    data = pickle.load(f)
    
# Extract signals and ground truth HR values
signals = data["signals"]
ground_truth_hr = data["hr_values"]

print(f"Loaded {len(signals)} ECG signals.")

Loaded 200 ECG signals.


In [3]:
#print(signals)

In [4]:
# Checking the quality of the data
print(ground_truth_hr)
# for i in range(3):
   # plt.plot(signals[i][:1000])
   # plt.show()

[np.float64(60.56387051862165), np.float64(137.2351160443996), np.float64(52.97113752122241), np.float64(130.191557891168), np.float64(78.90743550834597), np.float64(100.5867560771165), np.float64(78.61582395430874), np.float64(80.13698630136984), np.float64(128.48402447314754), np.float64(76.88416793120892), np.float64(78.7746170678337), np.float64(62.81661600810537), np.float64(97.03504043126685), np.float64(74.60930935977146), np.float64(91.94619444917419), np.float64(59.10326086956521), np.float64(70.85793678360542), np.float64(79.4836956521739), np.float64(72.401310118945), np.float64(84.40555841482244), np.float64(126.08032536858158), np.float64(125.6763833129691), np.float64(57.17202654415518), np.float64(112.8653006382264), np.float64(108.71794871794872), np.float64(79.28172115873285), np.float64(81.46639511201629), np.float64(80.23315618035316), np.float64(53.324218082379076), np.float64(121.52420185375901), np.float64(116.5829145728643), np.float64(75.06339814032121), np.floa

# --- 2. Implement Your HR Extraction Algorithm Here ---

**Instruction:**

***Your algorithm should return a list of HR values where each HR value corresponds to an ECG signal. Ensure the length of the list is 200 (equal to number of signals). The list can include np.nan if your algorithm is not able to calculate HR for a signal.***

In [5]:
# Helper functions


# Butterworth filter (bandpass, lowpass or highpass)
def butterworth_filter(s, fs, lc=0.5, hc=20, type='band'):
    if type == 'band':
        sos = butter(4, [lc, hc], btype='bandpass', fs=fs, output='sos')
    if type == 'high':
        sos = butter(4, [lc], btype='highpass', fs=fs, output='sos')
    if type == 'low':
        sos = butter(4, [hc], btype='lowpass', fs=fs, output='sos')
    return sosfiltfilt(sos, s) 

#derivative filter
#compute the derivate of the given signal to obtain information
# about the slope of the qrs
def derivative(s, fs):
    
    b= np.array([1,2,0,-2,-1]) * (1/8) * fs #numerator
    a= 1 #denomitor
    sig= filtfilt(b, a, s)
    return sig / max(sig)

#squaring
def square(s):
    squared= s**2
    return squared

#moving window
def ma(s, fs):
    averaged= np.convolve(s, triang(int(fs *0.1)), mode='same')
    return averaged

# Normalization  
def minmax_normalize(s, min_value: int = 0, max_value: int = 1):
    s = (s - np.min(s)) / (np.max(s) - np.min(s))
    return s * (max_value - min_value) + min_value

# Compute timearray for the signal
def new_time_vector(s, fs):
    sampling_period = 1/fs
    time_vector = np.arange(len(s))*sampling_period
    return time_vector

def peaks_processing(peaks, kernel_size = 3, hr_max_diff = 16, hr_min = 40, hr_max = 180):
    peaks=np.asarray(peaks)
    # Convert to RR and HR
    rr = np.diff(peaks) / 200 # Fs
    hr_instant = 60 / rr

    # Apply median filter to smooth sudden peaks
    hr_med = medfilt(hr_instant, kernel_size)

    # Check physiological constraints
    valid_rr = (hr_instant  > hr_min) & (hr_instant  < hr_max) & (np.abs(hr_instant-hr_med) < hr_max_diff)
    # Convert the back to the peak indices (skip the first peak because it is not compared properly since there is not RR-interval before it)
    filtered_peaks = peaks[1:][valid_rr]
    
    return filtered_peaks

In [6]:
# Variables
FS = 200 
min_peak_distance = int(0.4*FS) # R-peaks can't be closer than 200ms 

def extract_hr(ecg_signals):
    """
    HR extraction algorithm by Angelina and Janita.
    """

    detected_hr_values = [np.nan] * 200  # Ensuring list has 200 elements
    for i, signal in enumerate(ecg_signals):  
        # Normalize
        normal = minmax_normalize(signal)
        # Preprosessing according to Pan-Tompkins
        # Bandpass filter
        bfiltered  = butterworth_filter(s=normal, fs=FS, lc=0.5, hc=40, type='band')
        # Derivative filter
        deriv = derivative(s=bfiltered, fs=FS)
        # Squaring
        sq = square(deriv)
        # Moving average
        moving_avg = ma(s=sq, fs=FS)
        # Normalized
        normalized = minmax_normalize(moving_avg)
        # Peak detection
        peaks, _ = find_peaks(normalized, distance=min_peak_distance, height = 0.1)
        # Filter physologically plausible peaks
        filtered_peaks = peaks_processing(peaks)
        # Calculate HR
        RR = np.diff(filtered_peaks) / FS
        HR = 60/np.mean(RR)
        # print(HR)
        
        detected_hr_values[i]=HR

    return detected_hr_values

# Run your HR extraction algorithm
detected_hr_values = extract_hr(signals)

# --- 3. Evaluation of HR Extraction Algorithm ---

**Description:**

***Evaluates the performance of your HR extraction method using Mean Absolute Error (MAE).***

In [7]:
def evaluate_hr_extraction(detected_hr_values, ground_truth_hr):
    """
    Evaluates HR extraction performance using Mean Absolute Error (MAE).
    
    Parameters:
        - detected_hr_values (List of floats): List of detected HR values.
        - ground_truth_hr (List of floats): List of ground truth HR values.
    
    Returns:
        - MAE score.
    """
    valid_indices = np.where(~np.isnan(detected_hr_values))[0].tolist() 
    
    if len(valid_indices) == 0:
        return {"Mean Absolute Error": np.nan}
    
    absolute_errors = np.abs(np.array(ground_truth_hr)[valid_indices] - np.array(detected_hr_values)[valid_indices])
    mae = np.sum(absolute_errors) / len(absolute_errors)
    return {"Mean Absolute Error": mae}

In [8]:
# Evaluate performance of your HR extraction method
mae = evaluate_hr_extraction(detected_hr_values, ground_truth_hr)

print(mae)

{'Mean Absolute Error': np.float64(7.944561816410917)}


***Run the above cells and check your evaluation score.***

***The final assessment is based on the MAE, the lowest MAE is the first rank in competition!***